# Feature Importance & Erklärbarkeit der Referenz-SVM (#34)

**Frage:** Welche Wellenlängen (und welche Metadaten) braucht die Referenz-SVM, um Kultur und Wachstumsstadium zu erkennen – und passt das zur Domäne?

**Kurzfazit für das Statusmeeting**

- **Referenz:** kombinierte RBF-SVM `combined_svm_expanded` (`C=100`, `gamma=0.003`), CV-BAcc kombiniert 0.847 (Kultur 0.925, Stadium 0.909). Analysiert wird nur der Trainingsanteil; der Validierungsanteil bleibt unberührt.
- **Wichtigste Gruppen** (Abfall der kombinierten BAcc, wenn die Gruppe gemischt wird): **NIR 0.47** und der **Monat 0.44**, danach **SWIR 1 0.33** und **SWIR 2 0.32**. Rot, Red Edge, Blau und AEZ liegen bei etwa 0.2, Grün bei 0.17.
- **Einzelbänder** sind einzeln kaum entscheidend (höchstens 0.04): Das Spektrum ist stark redundant. Als einziger Bereich hebt sich die **Red Edge (712–742 nm)** konsistent ab.
- **Je Kultur und Stadium** unterscheiden sich die wichtigsten Bereiche deutlich: Mais und Baumwolle hängen am SWIR 2, Reis, Soja und Winterweizen am NIR. Frühe Stadien brauchen NIR und SWIR 1, mittlere Stadien vor allem den Monat.
- **SHAP bestätigt das Bild** (Rangkorrelation 0.95 mit der Permutation Importance).
- **Offen:** Abgleich mit der Bandauswahl (#126) und der Bandreduktionsstudie (#33), beide noch nicht bearbeitet.

## Vorgehen

| Schritt | Methode | Warum |
| --- | --- | --- |
| 1 | **Permutation Importance** (modellunabhängig) auf Regionen und auf Einzelbändern | funktioniert für die SVM ohne Änderung am Modell |
| 2 | Aufschlüsselung **je Kultur und Stadium** | Aufgabenstellung: „welche Wellenlängen für welche Crops/Stages“ |
| 3 | **SHAP** (KernelSHAP) als zweite, unabhängige Methode | prüft, ob die Permutation Importance trägt |
| 4 | Einordnung mit **Domänen-Doku** und **Index-Katalog** (#123) | Ergebnisse fachlich begründen |

**Integrated Gradients** (die zweite Methode der Folie) braucht ein differenzierbares Modell. Die Referenz-SVM ist es nicht, und die neuronalen Modelle (MLP, CNN, LSTM) liegen in der Güte deutlich darunter. Eine Erklärung dieser Modelle würde nicht die Referenz erklären; wir verzichten deshalb darauf.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from scipy.stats import spearmanr
from shap.utils._legacy import DenseData

from awp2.config import (
    BAND_GROUPING,
    BAND_PREFIX,
    CROP_KIND,
    CROPS,
    INDEX_WAVELENGTH_TOLERANCE_NM,
    METRIC_KIND,
    MODEL_DOCS_FIGURES_DIR,
    PERMUTATION_REPEATS,
    REGION_GROUPING,
    SEED,
    SPECTRAL_REGIONS,
    STAGE_KIND,
    STAGES,
    VAL_SIZE,
    VEGETATION_INDEX_SPECS,
)
from awp2.data import combined_label, load_split, load_split_config, wavelength
from awp2.evaluation import METRIC_LABELS
from awp2.experiment import RunConfig
from awp2.explain import feature_groups, permutation_importance
from awp2.models import make_combined_svm
from awp2.plots import plot_importance_heatmap, plot_importance_spectrum, save_doc_figure
from awp2.preprocessing import PreprocessingConfig, build_preprocessor

# Der Referenzlauf `combined_svm_expanded` nutzt das bewährte `scale=True`.
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Referenzmodell und Datenbasis

- Modell: die per CV gewählte RBF-SVM `combined_svm_expanded` mit der Vorverarbeitung des Referenzlaufs (Spektren und Monat standardisiert, AEZ one-hot, Monat zyklisch). Die Hyperparameter stammen aus `combined_svm_expanded_search` (MLflow `0dea8124`) und werden hier nicht neu gesucht.
- Daten: ausschließlich der **Trainingsanteil** des gemeinsamen Splits (70/30, Prüfung per `assert`). Die Permutation Importance läuft über die fünf gemeinsamen CV-Folds: je Fold wird das Modell auf den vier Trainingsfolds gefittet und auf dem fünften bewertet. Der Validierungsanteil wird nicht verwendet, damit die Ergebnisse später für Merkmalsentscheidungen genutzt werden dürfen, ohne ihn zu verbrauchen.
- Die Tabelle unten zeigt die **Merkmalsgruppen** nach der Vorverarbeitung: 140 Merkmale, davon 131 Spektralbänder in sieben Spektralregionen (Grenzen aus der [Domänen-Doku](../docs/domaene/vegetation.md)) sowie AEZ (7 One-Hot-Spalten) und Monat (Sinus/Kosinus). Die 67 leeren Bänder sind bereits entfernt.

In [ ]:
# Gewinner von `combined_svm_expanded_search` (MLflow 0dea8124), per CV gewählt
REFERENCE_PREPROCESSING = PreprocessingConfig(scale=True, use_meta=True)
REFERENCE_PARAMS = {"estimator__C": 100.0, "estimator__gamma": 0.003}

reference_model = make_combined_svm().set_params(**REFERENCE_PARAMS)
reference_config = RunConfig(
    name="combined_svm_importance",
    approach="combined",
    preprocessing=REFERENCE_PREPROCESSING,
    track=False,
)

split = load_split()
assert load_split_config().validation_size == VAL_SIZE, (
    "Die Referenzwerte gelten für den Standard-Split; erzeuge ihn mit `make data`."
)
features = build_preprocessor(REFERENCE_PREPROCESSING).fit_transform(split.X_train)
region_groups = feature_groups(list(features.columns), REGION_GROUPING)

print(f"Trainingsanteil: {len(split.X_train)} Zeilen, {features.shape[1]} Merkmale")
pd.Series(
    {name: len(columns) for name, columns in region_groups.items()}, name="Merkmale je Gruppe"
).to_frame().T

## 1. Permutation Importance der Regionen

**Idee:** Die Werte einer Merkmalsgruppe werden im Validierungsfold zeilenweise vertauscht. Das Modell bleibt unverändert; sinkt die Balanced Accuracy stark, nutzt es diese Gruppe stark.

$$\text{Wichtigkeit}_g = \text{BAcc}_{\text{ohne Mischen}} - \text{BAcc}_{g \text{ gemischt}}$$

- Gruppen: sieben Spektralregionen, AEZ und Monat (AEZ-Spalten und Sinus/Kosinus werden jeweils **gemeinsam** gemischt).
- 5 Folds × 5 Wiederholungen = 25 Werte je Gruppe; angegeben sind Mittelwert ± Standardabweichung.
- Gemessen wird die BAcc für Kultur, Stadium und die Kombination – nie nur die einfache Accuracy.

In [ ]:
N_REPEATS_REGION = 5

regions = permutation_importance(
    reference_model, reference_config, REGION_GROUPING, n_repeats=N_REPEATS_REGION
)

metric_mean = regions.mean[METRIC_KIND].rename(columns=METRIC_LABELS)
metric_std = regions.std[METRIC_KIND].rename(columns=METRIC_LABELS)
print("Baseline (ohne Mischen, Mittel über die Folds):")
display(regions.baseline[METRIC_KIND].rename(METRIC_LABELS).round(3).to_frame("BAcc").T)

summary = metric_mean.round(3).astype(str) + " ± " + metric_std.round(3).astype(str)
summary.sort_values(METRIC_LABELS["bacc_combined"], ascending=False)

In [ ]:
region_fig, ax = plt.subplots(figsize=(10, 4.5))
metric_mean.plot.bar(ax=ax, yerr=metric_std, rot=0, capsize=2, width=0.8)
ax.set_xlabel("")
ax.set_ylabel("Abfall der Balanced Accuracy beim Mischen")
ax.set_title("Permutation Importance der Spektralregionen und Metadaten (Referenz-SVM)")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize="small")
region_fig.tight_layout()

### Ergebnis: Regionen

- **NIR (0.47) und Monat (0.44)** sind am wichtigsten für die kombinierte BAcc, gefolgt von **SWIR 1 (0.33)** und **SWIR 2 (0.32)**. Rot, Red Edge, Blau und AEZ liegen zwischen 0.20 und 0.22, Grün bei 0.17 – der Abstand zwischen den Gruppen übersteigt die Standardabweichung (0.02–0.03) nur bei NIR, Monat und den beiden SWIR-Regionen klar.
- **Kultur und Stadium nutzen unterschiedliche Informationen:** Für die Kultur zählt nach dem NIR (0.43) vor allem das SWIR (je 0.28), für das Stadium nach dem NIR (0.42) der **Monat (0.41)**; der Monat hilft für die Kultur kaum (0.21).
- **Der Monat ist ein Kontextmerkmal, kein Spektralmerkmal.** Dass er fast so wichtig ist wie das NIR, passt zu den Beobachtungen in der [Domänen-Doku](../docs/domaene/kulturpflanzen.md) (Stadien liegen in festen Monaten) und zur früheren Messung ohne Metadaten (CV 0.742 gegenüber 0.834). Die Aussage „welche Wellenlängen sind wichtig“ gilt daher **zusätzlich** zum Monat.
- **Regionen sind unterschiedlich groß** (NIR 44 Bänder, SWIR 29 und 27, alle anderen 7–9). Ein Teil des Vorsprungs von NIR und SWIR kommt daher, dass dort mehr Bänder gleichzeitig zerstört werden; pro Band sind die schmalen Bereiche dichter (Red Edge: 7 Bänder, Abfall 0.21).

## 2. Permutation Importance der Einzelbänder

Hier wird jedes der 131 Bänder allein gemischt (5 Folds × 3 Wiederholungen). Benachbarte Bänder sind stark korreliert; das Modell kann ein gemischtes Band oft aus seinen Nachbarn ersetzen. Einzelwerte **unterschätzen** daher die Wichtigkeit einer Region. Sie zeigen aber, **wo** im Spektrum die Information konzentriert ist. Lücken im Verlauf sind die entfernten Wasserabsorptions- und Randbänder.

In [ ]:
bands = permutation_importance(
    reference_model, reference_config, BAND_GROUPING, n_repeats=PERMUTATION_REPEATS
)
band_names = [name for name in bands.mean.index if name.startswith(BAND_PREFIX)]
combined_key = (METRIC_KIND, "bacc_combined")
band_importance = bands.mean.loc[band_names, combined_key]
band_std = bands.std.loc[band_names, combined_key]

band_ax = plot_importance_spectrum(
    band_importance,
    band_std,
    title="Permutation Importance einzelner Bänder (BAcc kombiniert, Referenz-SVM)",
)
band_fig = band_ax.figure

In [ ]:
def region_of(nm: float) -> str:
    return next((r.name for r in SPECTRAL_REGIONS if r.start_nm <= nm < r.end_nm), "-")


implemented_indices = [spec for spec in VEGETATION_INDEX_SPECS if spec.status == "implemented"]


def catalogue_indices_near(nm: float) -> str:
    near = [
        spec.name
        for spec in implemented_indices
        if any(
            abs(target - nm) <= INDEX_WAVELENGTH_TOLERANCE_NM
            for target in spec.target_wavelengths_nm
        )
    ]
    return ", ".join(near) or "-"


top_bands = band_importance.sort_values(ascending=False).head(15)
pd.DataFrame(
    {
        "Wellenlänge [nm]": [wavelength(band) for band in top_bands.index],
        "Region": [region_of(wavelength(band)) for band in top_bands.index],
        "Abfall BAcc kombiniert": top_bands.round(4).to_numpy(),
        "Std": band_std[top_bands.index].round(4).to_numpy(),
        "Katalog-Indizes (±5 nm)": [
            catalogue_indices_near(wavelength(band)) for band in top_bands.index
        ],
    },
    index=top_bands.index,
)

### Ergebnis: Einzelbänder

- Der größte Einzelwert ist 0.04 (Band `X722`) – weit unter dem Wert ganzer Regionen (bis 0.47). Die Information ist also **stark redundant** über benachbarte Bänder verteilt.
- Die Red Edge fällt als **einzige zusammenhängende Spitze** auf: `X712`, `X722`, `X732` und `X742` stehen alle unter den 15 wichtigsten Bändern (Maximum bei 722 nm). Hier steigt die Vegetationsreflexion steil an – die Lage dieses Anstiegs ist genau das, was die Red-Edge-Position (REIP) beschreibt.
- Viele andere Werte sind **nicht von Rauschen zu trennen:** Die Standardabweichung über die 15 Werte je Band (5 Folds × 3 Wiederholungen) liegt bei 0.01–0.02 und damit in der Größe des Mittelwertes. Das gilt besonders für die blauen Bänder `X437` und `X447` (Standardabweichung 0.023 und 0.024).
- `X1316` (am Rand eines entfernten Blocks) und `X2345` (Ende des Spektrums) stehen auffällig weit oben. Das kann ein **Randeffekt** sein; wir deuten diese Bänder fachlich nicht.
- Nur drei der 15 wichtigsten Bänder liegen innerhalb von ±5 nm einer Wellenlänge des Index-Katalogs (`X447`, `X752`, `X763`).

## 3. Je Kultur und je Stadium

Gleiche Permutation der Regionen, aber bewertet wird der **Recall** der einzelnen Kultur bzw. des einzelnen Stadiums (Zeilen: gemischte Gruppe; Spalten: Klasse). Ein hoher Wert heißt: Ohne diese Gruppe findet das Modell die Klasse deutlich schlechter.

In [ ]:
crop_importance = regions.mean[CROP_KIND][
    [crop for crop in CROPS if crop in regions.mean[CROP_KIND]]
]
stage_importance = regions.mean[STAGE_KIND][
    [stage for stage in STAGES if stage in regions.mean[STAGE_KIND]]
]

crop_ax = plot_importance_heatmap(
    crop_importance, "Abfall des Recalls je Kultur beim Mischen einer Region"
)
crop_fig = crop_ax.figure
stage_ax = plot_importance_heatmap(
    stage_importance, "Abfall des Recalls je Stadium beim Mischen einer Region"
)
stage_fig = stage_ax.figure

top_region = pd.concat([crop_importance, stage_importance], axis=1).idxmax()
top_region.rename("wichtigste Gruppe").to_frame().T

### Ergebnis: je Kultur und Stadium

**Kulturen**

- **Mais:** SWIR 2 (0.39), dann SWIR 1 (0.27) und NIR (0.21). Der Monat spielt kaum eine Rolle (0.13).
- **Baumwolle:** SWIR 2 (0.50), Monat (0.39) und Rot (0.38).
- **Reis:** NIR mit 0.78 – der größte Einzelwert der Abbildung –, dahinter SWIR 1 (0.41) und AEZ (0.40). Reis hat nur 93 Beobachtungen (rund 13 je Validierungsfold); die Werte schwanken stärker als bei den anderen Kulturen.
- **Soja:** NIR (0.38), Red Edge (0.29), Grün (0.26).
- **Winterweizen:** NIR (0.46), SWIR 1 (0.41), AEZ (0.32).

**Stadien**

- **`Emerge_VEarly`:** NIR (0.52), SWIR 1 (0.43), Monat (0.38) – passend zu einem lückigen Bestand, in dem Boden und Pflanze gemischt sind (siehe [Domänen-Doku](../docs/domaene/vegetation.md)).
- **`Early_Mid` und `Critical`:** vor allem der **Monat** (0.44 und 0.35); spektral SWIR 2 (0.35) bzw. NIR (0.27).
- **`Late` und `Harvest`:** NIR (0.57 und 0.54) und Monat (0.49 und 0.51).
- **`Mature_Senesc`:** NIR (0.37), SWIR 2 (0.28) und SWIR 1 (0.26) – die Domänen-Doku erwartet bei alternden Beständen Änderungen von Blattstruktur und Wassergehalt im NIR/SWIR.
- **Sichtbar und Red Edge** (Rot, Red Edge, Blau, Grün) spielen für die Stadien mit höchstens 0.20 nur eine Nebenrolle; die Hypothese der Domänen-Doku (Chlorophyll an Red Edge und Rot trennt grünen Bestand von Seneszenz) wird damit **nicht** deutlich gestützt.

## 4. SHAP als zweite Methode

**KernelSHAP** erklärt, wie viel jede Gruppe zum Modell-Score einer Klasse beiträgt, im Vergleich zu einem Hintergrund aus zehn Cluster-Zentren des Trainingsanteils.

- Erklärt wird `decision_function` der SVM (ein Score je Klasse, 23 Klassen).
- Die Gruppen sind dieselben neun wie oben. Bei neun Gruppen sind nur 2⁹ = 512 Koalitionen nötig, die **Shapley-Werte sind damit exakt** (statt einer Näherung über 140 Einzelmerkmale).
- Stichprobe: bis zu 10 Zeilen je kombinierter Klasse aus dem Trainingsanteil (228 Zeilen). Verwendet wird für jede Zeile der Beitrag zum Score der **eigenen** Klasse; gemittelt wird der Betrag.
- Das Modell wird hier auf dem gesamten Trainingsanteil gefittet. Es ist eine Erklärung des Modells, keine Bewertung – der Validierungsanteil wird nicht verwendet.

In [ ]:
SHAP_SAMPLES_PER_CLASS = 10
SHAP_BACKGROUND_CLUSTERS = 10

model = make_combined_svm().set_params(**REFERENCE_PARAMS).fit(features, split.y_train)
svc = model.estimator_

# Exakte Shapley-Werte über 9 Gruppen (2^9 Koalitionen) statt Näherung über 140 Einzelmerkmale.
group_names = list(region_groups)
group_positions = [
    np.array([features.columns.get_loc(column) for column in columns])
    for columns in region_groups.values()
]
background = shap.kmeans(features.to_numpy(), SHAP_BACKGROUND_CLUSTERS)
explainer = shap.KernelExplainer(
    svc.decision_function, DenseData(background.data, group_names, group_positions)
)

pair_labels = combined_label(split.y_train)
sample_index = (
    pair_labels.groupby(pair_labels, group_keys=False)
    .apply(lambda rows: rows.sample(n=min(SHAP_SAMPLES_PER_CLASS, len(rows)), random_state=SEED))
    .index
)
sample_positions = features.index.get_indexer(sample_index)
print(f"{len(sample_index)} Zeilen erklärt, {len(svc.classes_)} Klassen")

In [ ]:
shap_values = explainer.shap_values(
    features.iloc[sample_positions].to_numpy(), nsamples="auto", silent=True
)
shap_values.shape  # Zeilen x Gruppen x Klassen

In [ ]:
class_index = {label: number for number, label in enumerate(svc.classes_)}
sample_pairs = pair_labels.loc[sample_index]
own_class = np.array([class_index[label] for label in sample_pairs])
own_class_values = shap_values[np.arange(len(sample_pairs)), :, own_class]

shap_abs = pd.DataFrame(np.abs(own_class_values), columns=group_names, index=sample_index)
sample_targets = split.y_train.loc[sample_index]
shap_crop = shap_abs.groupby(sample_targets["Crop"].to_numpy()).mean().T[list(CROPS)]
shap_stage = shap_abs.groupby(sample_targets["Stage"].to_numpy()).mean().T[list(STAGES)]

shap_crop_ax = plot_importance_heatmap(
    shap_crop, "Mittlerer |SHAP-Wert| der eigenen Klasse je Kultur"
)
shap_crop_ax.figure.axes[-1].set_ylabel("mittlerer |SHAP-Wert|")
shap_crop_fig = shap_crop_ax.figure
shap_stage_ax = plot_importance_heatmap(
    shap_stage, "Mittlerer |SHAP-Wert| der eigenen Klasse je Stadium"
)
shap_stage_ax.figure.axes[-1].set_ylabel("mittlerer |SHAP-Wert|")
shap_stage_fig = shap_stage_ax.figure

In [ ]:
permutation_share = regions.mean[combined_key].clip(lower=0)
permutation_share = permutation_share / permutation_share.sum()
shap_share = shap_abs.mean() / shap_abs.mean().sum()

comparison = pd.DataFrame({"Permutation Importance": permutation_share, "SHAP": shap_share})
rank_correlation = spearmanr(comparison["Permutation Importance"], comparison["SHAP"]).statistic

compare_fig, ax = plt.subplots(figsize=(10, 4))
comparison.plot.bar(ax=ax, rot=0, width=0.8)
ax.set_ylabel("Anteil an der Gesamtwichtigkeit")
ax.set_title(f"Permutation Importance und SHAP im Vergleich (Spearman {rank_correlation:.2f})")
ax.grid(axis="y", alpha=0.3)
compare_fig.tight_layout()
comparison.round(3)

### Ergebnis: SHAP

- **Rangfolge:** Permutation Importance und SHAP ordnen die neun Gruppen fast gleich (Spearman 0.95). Unterschiede: SHAP setzt Monat (0.24) und NIR (0.235) etwas höher als die Permutation (0.17 und 0.185); SWIR 1 und SWIR 2 sind in beiden Verfahren gleich (je ≈ 0.13).
- **Je Kultur:** Der größte Wert ist wie bei der Permutation Reis/NIR (9.1). Der Monat dominiert bei Winterweizen (4.8), Baumwolle (3.7) und Mais (3.1).
- **Je Stadium:** `Late` und `Harvest` hängen am NIR (5.4 und 4.5), `Emerge_VEarly` am Monat (4.9).
- **Wo sich die Verfahren unterscheiden:** Für Mais ist bei der Permutation das SWIR 2 vorn (0.39), bei SHAP der Monat (3.05). Das liegt auch an der Messgröße: SHAP misst den Beitrag zum Score, die Permutation den Verlust an Recall. Mit höchstens 10 Zeilen je Klasse sind die klassenweisen SHAP-Werte zudem ungenauer als die gruppierten.

## 5. Einordnung mit Domäne und Index-Katalog

Der Index-Katalog (#123) nutzt die Wellenlängen der Kickoff-Indizes. Die Tabelle zählt, wie viele der berechenbaren Indizes mindestens ein Band in einer Region haben (ein Index kann mehrere Regionen berühren), und stellt dem die Wichtigkeit der Region gegenüber. Die Abbildung legt die Katalog-Wellenlängen (graue Linien) über die Einzelband-Importance.

In [ ]:
catalogue = pd.DataFrame(
    [
        {
            "Index": spec.name,
            "Kategorie": spec.category,
            "Regionen": ", ".join(
                sorted({region_of(target) for target in spec.target_wavelengths_nm})
            ),
        }
        for spec in implemented_indices
    ]
)
catalogue_regions = (
    pd.Series([region for cell in catalogue["Regionen"] for region in cell.split(", ")])
    .value_counts()
    .reindex([region.name for region in SPECTRAL_REGIONS], fill_value=0)
)
region_overview = pd.DataFrame(
    {
        "Katalog-Indizes mit Band in der Region": catalogue_regions,
        "Abfall BAcc kombiniert": regions.mean.loc[catalogue_regions.index, combined_key].round(3),
    }
)
region_overview

In [ ]:
index_wavelengths = sorted(
    {target for spec in implemented_indices for target in spec.target_wavelengths_nm}
)
band_catalogue_ax = plot_importance_spectrum(
    band_importance,
    band_std,
    title="Bänder-Importance mit den Wellenlängen der Katalog-Indizes (graue Linien)",
)
band_catalogue_ax.vlines(
    index_wavelengths,
    *band_catalogue_ax.get_ylim(),
    color="0.5",
    linewidth=0.8,
    alpha=0.6,
    zorder=1,
)
band_catalogue_fig = band_catalogue_ax.figure

### Ergebnis: Bezug zur Domäne

- **Red Edge:** Die Spitze bei 712–742 nm entspricht dem steilen Anstieg zwischen Rot und NIR (Red-Edge-Position). Der Katalog nutzt die Red Edge am häufigsten (16 Indizes), aber nur als Bandpaare mit Wellenlängen von 680 bis 705 nm und 750 nm; kein Katalog-Index liegt auf der wichtigsten Wellenlänge 722 nm. Die REIP-Merkmale des Katalogs beschreiben dagegen die Lage des Anstiegs.
- **NIR und Wasser:** Im NIR liegen mehrere wichtige Bänder bei 1175–1255 nm, nahe den Wellenlängen der Katalog-Indizes `NDWI_1070_1200` und `NDWI_860_1240` (die zweite entspricht der NDWI-Variante nach Gao, siehe [Vegetation](../docs/domaene/vegetation.md#indizes-fur-photosynthese-wasser-und-trockene-bestandteile)). Das ist ein Hinweis auf einen Wasserbezug, aber kein Beleg: Eine Zuordnung zur Blattwasser-Absorption müsste noch mit Literatur belegt werden.
- **SWIR 2 ohne Katalog-Index:** Die Region hat 0 Katalog-Indizes, gehört aber zu den drei wichtigsten Bereichen (0.32; für Mais und Baumwolle die wichtigste). Der naheliegende Index, CAI (trockene Pflanzenbestandteile um 2100 nm), ist wegen des leeren Bandes `X2002` nicht direkt berechenbar. Wichtige Bänder liegen bei 2063, 2123 und 2184 nm. Ob eine angepasste CAI-Variante hilft, ist eine offene, prüfbare Hypothese.
- **Wasserabsorptionsbänder** der Atmosphäre (um 1400 und 1900 nm) sind bereits entfernt und können nicht als „wichtig“ erscheinen; sie sind als Lücken im Verlauf sichtbar.
- **Chlorophyll (Blau, Rot, Grün):** gering bis mittel (0.17–0.22), obwohl die meisten Katalog-Indizes dort Bänder haben (Grün 10, Rot 7). Die Indizes drücken die Information in diesen Bereichen offenbar anders aus als das Modell sie nutzt – oder das NIR/SWIR trägt die Trennung.

## Grenzen und offene Punkte

- **Korrelierte Bänder:** Einzelband-Permutation unterschätzt die Wichtigkeit; Regionen überschätzen sie teilweise über die Anzahl der Bänder. Beide Sichten gemeinsam lesen.
- **Permutation außerhalb der Verteilung:** Gemischte Zeilen sind keine realistischen Spektren; die Werte sind Hinweise auf die Modellnutzung, kein Kausalnachweis.
- **Metadaten:** Monat und AEZ gehören zur Referenz und sind stark gewichtet. „Wichtige Wellenlänge“ heißt hier „wichtig **im Zusammenspiel** mit dem Kontext“. Die spektrale Wichtigkeit ohne Metadaten (Modell `combined_svm_no_meta`) wurde nicht untersucht.
- **Kleine Klassen:** Reis (93 Beobachtungen) und seltene Stadien schwanken stärker; SHAP nutzt höchstens 10 Zeilen je Klasse.
- **Nur eine Referenz:** Die Aussagen gelten für die RBF-SVM; andere Modelle können andere Bereiche nutzen.
- **Bandauswahl (#126) und Studie (#33):** Beide sind noch offen. Der Abgleich („entbehrliche Bänder dürfen hier nicht als wichtig erscheinen“) ist erst möglich, wenn deren Ergebnisse vorliegen. Eine eigene Bandauswahl nach dieser Rangliste wäre mit denselben Folds Leakage; sie muss innerhalb der Folds geschehen.

In [ ]:
for name, figure in {
    "feature_importance_regionen": region_fig,
    "feature_importance_baender": band_fig,
    "feature_importance_kultur": crop_fig,
    "feature_importance_stadium": stage_fig,
    "feature_importance_shap_kultur": shap_crop_fig,
    "feature_importance_shap_stadium": shap_stage_fig,
    "feature_importance_vergleich": compare_fig,
    "feature_importance_katalog": band_catalogue_fig,
}.items():
    save_doc_figure(figure, name, MODEL_DOCS_FIGURES_DIR)

## Zusammenfassung

1. **Referenz-SVM:** wichtig sind vor allem **NIR** und der **Monat**, danach **SWIR 1 und SWIR 2**; sichtbare Bereiche und Red Edge tragen jeweils weniger bei.
2. **Je Kultur/Stadium:** SWIR 2 für Mais und Baumwolle, NIR für Reis, Soja und Winterweizen; frühe Stadien über NIR und SWIR 1, mittlere über den Monat, späte über NIR und Monat.
3. **Einzelbänder:** viel Redundanz; die **Red Edge (712–742 nm)** ist der einzige klar abgrenzbare Bereich.
4. **Zwei Verfahren, ein Bild:** Permutation Importance und SHAP stimmen in der Rangfolge überein (Spearman 0.95).
5. **Fachlich:** Red Edge, NIR/Wasser und SWIR 2 sind plausibel mit der Domäne vereinbar; SWIR 2 ist im Index-Katalog nicht vertreten.

**Nächste Schritte:** Abgleich mit #126 und #33 sobald deren Ergebnisse vorliegen; Dokumentationsseite unter `docs/modelle/` mit den Abbildungen aus `docs/modelle/img/`; optional die spektrale Wichtigkeit ohne Metadaten.